In [ ]:
"""
finetune_gemma3n_kcc100k.py

Fine-tunes Gemma-3n (4B) on the `skylord/kcc-sample100k` dataset (train split only),
using Unsloth + LoRA + TRL's SFTTrainer. Adapted from the original FarmerApp
Colab notebook (gemma3n_(4b)_finetuning_for_farmerapp.py) for local execution.

Requires a CUDA GPU and the Unsloth stack:
  pip install unsloth bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
  pip install sentencepiece protobuf datasets huggingface_hub hf_transfer python-dotenv
  pip install transformers==4.57.3 trl==0.22.2 --no-deps
  pip install --no-deps --upgrade timm   # needed for Gemma 3N

Auth: put HF_TOKEN=hf_... in .env (already present in this repo) for push_to_hub.
"""

In [ ]:
import torch, re; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)

In [ ]:
xformers = "xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3")

In [ ]:
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo

In [ ]:
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

In [ ]:
!pip install --no-deps unsloth

In [ ]:
!pip install transformers==4.57.3

In [ ]:
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install torchcodec

In [ ]:
!pip install --no-deps --upgrade timm

In [ ]:
!pip install python-dotenv

In [ ]:
import os
import torch

from google.colab import drive
drive.mount('/gdrive')

In [ ]:
from google.colab import userdata

# Retrieve the secret token
token = userdata.get('HF_TOKEN')


In [ ]:
# ---------- CONFIG ----------
BASE_MODEL    = "unsloth/gemma-3n-E2B-it"
DATASET_REPO  = "skylord/kcc-sample100k"
MAX_SEQ_LEN   = 2048
OUTPUT_DIR    = "/gdrive/MyDrive/models/checkpoints-gemma-3n-kcc-sample100k"
SAVE_DIR      = "/gdrive/MyDrive/models/gemma-3n-kcc-sample100k"
PUSH_REPO_ID  = "skylord/gemma-3n-kcc-sample100k"   # set to None to skip push_to_hub
# ----------------------------


In [ ]:
## THIS CONFIG WAS ADDED TO RESUME THE TRAINING
### AFTER A CRASH OCCURRED AT 6400+ steps
import os
os.environ["TORCH_COMPILE"] = "0"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import torch
torch._dynamo.config.disable = True

!rm -rf /content/unsloth_compiled_cache /tmp/torchinductor_*
## THIS CONFIG WAS ADDED TO RESUME THE TRAINING
### AFTER A CRASH OCCURRED AT 6400+ steps

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    dtype = None,                # None for auto detection
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,         # 4-bit quantization to reduce memory
    full_finetuning = False,
)

In [ ]:
# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only finetune
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 4,
    lora_alpha = 4,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

In [ ]:
# ---------- Data prep ----------
from datasets import load_dataset

dataset = load_dataset(DATASET_REPO, split = "train")
print(dataset)
print(dataset[0])

In [ ]:
def formatting_prompts_func(examples):
    queries = examples["QueryText"]
    answers = examples["KccAns"]

    texts = []
    for query, answer in zip(queries, answers):
        convo = [
            {"role": "user", "content": [{"type": "text", "text": query}]},
            {"role": "model", "content": [{"type": "text", "text": answer}]},
        ]
        formatted_text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        ).removeprefix("<bos>")
        texts.append(formatted_text)
    return {"text": texts}


In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)
print(dataset[0]["text"])

In [ ]:
import numpy as np, random

def n_tokens(q, a):
    convo = [
        {"role": "user",  "content": [{"type": "text", "text": q}]},
        {"role": "model", "content": [{"type": "text", "text": a}]},
    ]
    ids = tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=False
    )
    return len(ids)

idx  = random.sample(range(len(dataset)), 2000)
lens = np.array([n_tokens(dataset[i]["QueryText"], dataset[i]["KccAns"]) for i in idx])

for p in [50, 90, 95, 99, 100]:
    print(f"{p:>3}th percentile: {int(np.percentile(lens, p)):>5} tokens")
print(f"max: {lens.max()} | fraction over 2048: {(lens > 2048).mean()*100:.2f}%")

In [ ]:
print(dataset.column_names)
print(repr(dataset[0]["QueryText"]), repr(dataset[0]["KccAns"]))

In [ ]:

# ---------- Trainer ----------
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        # max_steps = 210,
        learning_rate = 1e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        max_grad_norm = 1.0,
        warmup_steps = 15,
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = OUTPUT_DIR,
        bf16 = True,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 3,
    ),
)


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)


In [ ]:
# ---------- Memory stats (pre-train) ----------
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# ---------- Train ----------
trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
!ls -la /gdrive/MyDrive/models/checkpoints-gemma-3n-kcc-sample100k/checkpoint-6400

In [ ]:
import json
trainer_json = "/gdrive/MyDrive/models/checkpoints-gemma-3n-kcc-sample100k/checkpoint-6200/trainer_state.json"
with open(trainer_json) as f:
    s = json.load(f)
print("Resuming from step:", s["global_step"], "of", s.get("max_steps", "?"))
print("Last logged loss:", s["log_history"][-1])

In [ ]:
PUSH_REPO_ID  = "skylord/gemma-3n-kcc-sample100k-step6300"

CKPT = "/gdrive/MyDrive/models/checkpoints-gemma-3n-kcc-sample100k/checkpoint-6300"  # your folder

model, tokenizer = FastModel.from_pretrained(
    model_name = CKPT,          # Unsloth reads the base model from adapter_config.json and applies the adapter
    max_seq_length = 2048,
    load_in_4bit = True,
)
# FastModel.for_inference(model)   # enables faster inference

In [ ]:
from google.colab import userdata

# Retrieve the secret token
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
model.push_to_hub(PUSH_REPO_ID, token = HF_TOKEN)
tokenizer.push_to_hub(PUSH_REPO_ID, token = HF_TOKEN)
print(f"Pushed model + tokenizer to https://huggingface.co/{PUSH_REPO_ID}")

In [ ]:
# ---------- Memory/time stats (post-train) ----------
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


In [ ]:
# ---------- Save ----------
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved model + tokenizer to {SAVE_DIR}")

if PUSH_REPO_ID:
    if not HF_TOKEN:
        raise SystemExit("HF_TOKEN not found in .env — cannot push_to_hub.")
    model.push_to_hub(PUSH_REPO_ID, token = HF_TOKEN)
    tokenizer.push_to_hub(PUSH_REPO_ID, token = HF_TOKEN)
    print(f"Pushed model + tokenizer to https://huggingface.co/{PUSH_REPO_ID}")


In [ ]:

# ---------- Quick inference smoke test ----------
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "What is the best way to treat mites in apple orchards?"}],
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 2048,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)
